# Phase 5 — RWF-2000 crop training and late fusion

Select Kaggle **T4 x2**, enable Internet, and attach: (1) RWF-2000 videos, (2) the Phase 3 interaction-crop `.npz` output preserving its folder structure, and optionally (3) the Phase 4 `best.pt` checkpoint as a Kaggle Dataset. If the baseline checkpoint is not attached, this runner trains the full-frame baseline from the phase-5 cache. The cache contains full and crop uint8 views plus per-candidate crop-found masks; the notebook trains a crop specialist and a two-stream learned late-fusion model, evaluates all three inputs on the same held-out test set, and writes an availability-stratified comparison.

The paired cache occupies about **11 GiB** for 1,986 clips. Decoding is capped at two processes; DDP uses all visible GPUs, two loader workers and two Torch threads per GPU process. Fusion uses batch size 4 per GPU to keep dual-stream activation memory bounded.


## 1. Clone the code, load Kaggle secrets, and inspect both GPUs

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception as exc:
    print('W&B secret unavailable; logger will use its fallback:', exc)

REPO_URL = 'https://github.com/Ayaan-Ali-Khan/Violence-Detection.git'
REPO_DIR = Path('/kaggle/working/Violence-Detection')
GIT_REF = 'main'  # Pin to a commit SHA for the final report run.
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_REF], check=True)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
GPU_COUNT = torch.cuda.device_count()
print('PyTorch:', torch.__version__, '| visible GPUs:', GPU_COUNT)
for gpu_id in range(GPU_COUNT):
    print(f'GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}')
if GPU_COUNT < 1:
    raise RuntimeError('Select a Kaggle GPU accelerator before continuing.')
if GPU_COUNT != 2:
    print('The notebook will use all visible GPUs; select T4 x2 for the planned run.')


## 2. Locate RWF videos and the Phase 3 crop caches

In [ ]:
import pandas as pd
import zipfile
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
print('Kaggle input mounts:', *INPUT_ROOT.iterdir(), sep='\n  ')
# Optional: paste the mounted crop dataset path here, ending at yolo11n or its parent.
# Example: Path('/kaggle/input/datasets/aakcodebreaker/<crop-dataset-slug>/yolo11n')
CROP_CACHE_INPUT = None
raw_candidates = sorted({
    p.parent.parent for p in INPUT_ROOT.rglob('Fight')
    if p.is_dir() and p.parent.name == 'train'
    and (p.parent.parent / 'val' / 'Fight').is_dir()
    and (p.parent.parent / 'train' / 'NonFight').is_dir()
})
print('RWF roots found:', *raw_candidates, sep='\n  ')
if len(raw_candidates) != 1:
    raise RuntimeError('Attach exactly one RWF-2000 input with train and val Fight/NonFight folders.')
RAW_DIR = REPO_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
raw_target = RAW_DIR / 'rwf2000'
if raw_target.is_symlink():
    raw_target.unlink()
elif raw_target.exists():
    if any(raw_target.iterdir()):
        raise FileExistsError(f'Inspect existing non-empty path before replacing: {raw_target}')
    raw_target.rmdir()
raw_target.symlink_to(raw_candidates[0], target_is_directory=True)

manifest = pd.read_csv('data/splits/rwf2000.csv')
first_rows = manifest.groupby('split', sort=False).first().reset_index()
crop_search_root = Path(CROP_CACHE_INPUT) if CROP_CACHE_INPUT else INPUT_ROOT
if CROP_CACHE_INPUT and not crop_search_root.exists():
    raise FileNotFoundError(f'CROP_CACHE_INPUT does not exist: {crop_search_root}')
crop_files = list(crop_search_root.rglob('*.crop.npz'))
crop_zip_files = []
if not crop_files:
    for archive_path in crop_search_root.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                if any(name.lower().endswith('.crop.npz') for name in archive.namelist()):
                    crop_zip_files.append(archive_path)
        except zipfile.BadZipFile:
            continue
if len(crop_zip_files) == 1:
    extract_root = Path('/kaggle/working/phase3_crop_cache_unpacked')
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(crop_zip_files[0]) as archive:
        for member in archive.infolist():
            target = (extract_root / member.filename).resolve()
            if target != extract_root.resolve() and extract_root.resolve() not in target.parents:
                raise ValueError(f'Unsafe path in crop-cache archive: {member.filename}')
        archive.extractall(extract_root)
    crop_search_root = extract_root
    crop_files = list(crop_search_root.rglob('*.crop.npz'))
print('Crop archives discovered:', len(crop_files))

def crop_path_candidates(root, row):
    relative = Path(row['relative_path'])
    if relative.parts[:2] == ('data', 'raw'):
        relative = Path(*relative.parts[3:])
    filename = f'{relative.name}.crop.npz'
    # Support the standard cache layout and a flattened upload layout.
    return [
        root / row['dataset'] / row['split'] / relative.parent / filename,
        root / row['dataset'] / relative.parent / filename,
    ]

crop_roots = set()
for crop_file in crop_files:
    for part_index, part in enumerate(crop_file.parts):
        if part == 'rwf2000':
            crop_roots.add(Path(*crop_file.parts[:part_index]))
matching_roots = []
root_diagnostics = {}
for root in crop_roots:
    matched = 0
    for row in first_rows.to_dict('records'):
        if any(path.is_file() for path in crop_path_candidates(root, row)):
            matched += 1
    root_diagnostics[str(root)] = f'{matched}/{len(first_rows)} split samples found'
    if matched == len(first_rows):
        matching_roots.append(root)
print('Candidate crop roots:', root_diagnostics)
if len(matching_roots) != 1:
    examples = [str(path) for path in crop_files[:5]]
    zips = [str(path) for path in crop_search_root.rglob('*.zip')][:10]
    raise RuntimeError(f'Could not identify exactly one complete RWF crop-cache root. Crop files found: {len(crop_files)}. Candidate roots: {root_diagnostics}. Example crop paths: {examples}. ZIP files visible: {zips}. Attach a dataset containing individual .crop.npz files or set CROP_CACHE_INPUT to its mounted folder.')
CROP_CACHE_DIR = matching_roots[0]
print('Crop cache root:', CROP_CACHE_DIR)


## 3. Build paired full/crop tensors under a tau/margin-tagged profile

The cache builder validates the crop archive's settings hash against `configs/preprocess_crops.yaml`. It stores candidate-level crop-found masks alongside the RGB uint8 tensors so later metrics can distinguish real interaction crops, mixed clips, and all-fallback clips.

In [ ]:
TENSOR_CACHE = Path('/kaggle/working/tensor_cache/rwf2000')
INPUT_TAG = 'rwf_fullcrop_f16_s160_tau2p00_m0p10_v1'
subprocess.run([
    sys.executable, 'scripts/cache_tensors.py',
    '--manifest', 'data/splits/rwf2000.csv',
    '--config', 'configs/preprocess_crops.yaml',
    '--model', 'x3d_s',
    '--cache-dir', str(TENSOR_CACHE),
    '--crop-cache-dir', str(CROP_CACHE_DIR),
    '--mode', 'both',
    '--workers', '2',
], check=True)


## 4. Verify all views, crop masks, and checksums

In [ ]:
REPORT_DIR = Path('/kaggle/working/tensor_cache/reports')
subprocess.run([
    sys.executable, 'scripts/verify_tensor_cache.py',
    '--cache-dir', str(TENSOR_CACHE),
    '--input-tag', INPUT_TAG,
    '--manifest', 'data/splits/rwf2000.csv',
    '--report-dir', str(REPORT_DIR),
], check=True)
print((REPORT_DIR / 'verification_report.md').read_text())


## 5. Select or train the full-frame baseline; train the crop specialist

In [ ]:
import shutil

EXPERIMENTS = Path('/kaggle/working/experiments/phase5')
MODEL_CHECKPOINTS = Path('/kaggle/working/model_checkpoints/phase5')
FULL_OUTPUT = EXPERIMENTS / 'x3d_s_rwf2000_full'
CROP_OUTPUT = EXPERIMENTS / 'x3d_s_rwf2000_crop'
FUSION_OUTPUT = EXPERIMENTS / 'x3d_s_rwf2000_late_fusion'
FULL_CHECKPOINT_DIR = MODEL_CHECKPOINTS / 'x3d_s_rwf2000_full'
CROP_CHECKPOINT_DIR = MODEL_CHECKPOINTS / 'x3d_s_rwf2000_crop'
FUSION_CHECKPOINT_DIR = MODEL_CHECKPOINTS / 'x3d_s_rwf2000_late_fusion'

# Optional Phase 4 checkpoint from an attached Kaggle Dataset.
FULL_BASELINE_CHECKPOINT = None

def launch_training(config, output_dir, checkpoint_dir, extra_args=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nnodes=1',
        f'--nproc_per_node={GPU_COUNT}', 'src/violence_detection/train.py',
        '--config', config, '--cache-dir', str(TENSOR_CACHE),
        '--output-dir', str(output_dir), '--checkpoint-dir', str(checkpoint_dir),
    ] + (extra_args or [])
    print('Launching:', command)
    subprocess.run(command, check=True)

if FULL_BASELINE_CHECKPOINT is None:
    print('No Phase 4 checkpoint supplied; training full-frame baseline on the paired cache.')
    launch_training('configs/train_x3d_phase5_full.yaml', FULL_OUTPUT, FULL_CHECKPOINT_DIR, ['--mode', 'full'])
    FULL_BASELINE_CHECKPOINT = FULL_CHECKPOINT_DIR / 'best.pt'
else:
    FULL_BASELINE_CHECKPOINT = Path(FULL_BASELINE_CHECKPOINT)
    if not FULL_BASELINE_CHECKPOINT.is_file():
        raise FileNotFoundError(FULL_BASELINE_CHECKPOINT)
    FULL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(FULL_BASELINE_CHECKPOINT, FULL_CHECKPOINT_DIR / 'best.pt')
    FULL_BASELINE_CHECKPOINT = FULL_CHECKPOINT_DIR / 'best.pt'

launch_training('configs/train_x3d_crop.yaml', CROP_OUTPUT, CROP_CHECKPOINT_DIR, ['--mode', 'crop'])
print('Full checkpoint:', FULL_BASELINE_CHECKPOINT)
print('Crop checkpoint:', CROP_CHECKPOINT_DIR / 'best.pt')

## 6. Train learned late fusion from the two specialist checkpoints

The fusion model has two independent X3D-S streams. It loads the full-frame and crop specialist checkpoints, concatenates their per-stream class logits, and trains a small fusion head plus both streams with a lower backbone learning rate. The smaller per-GPU batch controls memory use.

In [ ]:
launch_training(
    'configs/train_x3d_fusion.yaml',
    FUSION_OUTPUT,
    FUSION_CHECKPOINT_DIR,
    [
        '--mode', 'both',
        '--init-full-checkpoint', str(FULL_BASELINE_CHECKPOINT),
        '--init-crop-checkpoint', str(CROP_CHECKPOINT_DIR / 'best.pt'),
    ],
)

## 7. Evaluate all three inputs on the same held-out test clips

In [ ]:
EVALS = {
    'full_frame': (FULL_BASELINE_CHECKPOINT, 'configs/train_x3d_phase5_full.yaml', 'full', EXPERIMENTS / 'comparison' / 'full_frame'),
    'crop_only': (CROP_CHECKPOINT_DIR / 'best.pt', 'configs/train_x3d_crop.yaml', 'crop', EXPERIMENTS / 'comparison' / 'crop_only'),
    'late_fusion': (FUSION_CHECKPOINT_DIR / 'best.pt', 'configs/train_x3d_fusion.yaml', 'both', EXPERIMENTS / 'comparison' / 'late_fusion'),
}
for name, (checkpoint, config, mode, output) in EVALS.items():
    subprocess.run([
        sys.executable, 'src/violence_detection/evaluate.py',
        '--checkpoint', str(checkpoint), '--config', config,
        '--cache-dir', str(TENSOR_CACHE), '--input-tag', INPUT_TAG,
        '--mode', mode, '--include-crop-status', '--output-dir', str(output), '--split', 'test',
    ], check=True)
    print(name, (output / 'test_results.md').read_text())


## 8. Compare overall metrics and crop-found/fallback subgroups

In [ ]:
COMPARE_DIR = EXPERIMENTS / 'comparison'
subprocess.run([
    sys.executable, 'scripts/compare_input_modes.py',
    '--full-predictions', str(EVALS['full_frame'][3] / 'test_predictions.csv'),
    '--crop-predictions', str(EVALS['crop_only'][3] / 'test_predictions.csv'),
    '--fusion-predictions', str(EVALS['late_fusion'][3] / 'test_predictions.csv'),
    '--output-dir', str(COMPARE_DIR),
], check=True)
print((COMPARE_DIR / 'input_comparison.md').read_text())


## 9. Output inventory

Download `tensor_cache/rwf2000/<input-tag>/` for the full/crop tensors and masks, `tensor_cache/reports/verification_report.*`, all three `best.pt` model checkpoints and learning curves, and `experiments/comparison/` for the per-input and subgroup metrics.

In [ ]:
for root in [REPORT_DIR, EXPERIMENTS, MODEL_CHECKPOINTS]:
    print(f'\n{root}')
    if root.exists():
        for path in sorted(root.rglob('*')):
            if path.is_file():
                print(path, path.stat().st_size, 'bytes')